# VIREM-UVIF: Latent Operational-Regime Alignment for Renewable Energy Decision Intelligence

This notebook upgrades the VIREM-UVIF workflow by shifting the two-dataset analysis from raw cross-dataset prediction to **latent operational-regime alignment**.

The central idea is:

> Renewable-energy datasets may have incompatible raw targets, but they can still be compared through normalized operational regimes such as low generation, normal operation, high generation, surplus, stress, and instability.

The notebook includes:

1. Google Drive mounting before dataset access.
2. Required project folder structure under `/content/drive/MyDrive/Outputs/VIREM_UVIF`.
3. Primary dataset loading from `Renewable.csv`.
4. UniSolar external dataset discovery and harmonization.
5. Leakage-safe primary forecasting.
6. Raw external prediction for diagnostic purposes only.
7. Distribution-shift diagnostics.
8. Percentile-based latent operational-regime construction.
9. Internal vs external regime alignment.
10. VIREM-UVIF regime-aware decision layer.
11. Regime transition analysis.
12. Decision robustness and evidence-acquisition analysis.
13. Reproducibility manifest and notebook destination in `PYTHON_DIR`.

The purpose is to support a stronger Q1-level scientific narrative: **decision robustness under distribution shift through latent operational alignment**.

## 1. Google Drive Mount and Project Structure

In [ ]:
# ============================================================
# 1. Google Drive Mount and Project Structure
# ============================================================

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
    print('Google Drive mounted successfully.')
except Exception as e:
    IN_COLAB = False
    print('Google Colab environment not detected.')
    print('Running in local/Jupyter environment.')
    print(f'Details: {e}')

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FRAMEWORK_NAME = 'VIREM'
FRAMEWORK_FULL_NAME = 'Variational Intelligence for Renewable Energy Management'

# Requested project location
if IN_COLAB:
    BASE_DIR = Path('/content/drive/MyDrive/Outputs/VIREM_UVIF')
else:
    BASE_DIR = Path.cwd() / 'VIREM_UVIF'

FIG_DIR    = BASE_DIR / 'Figures'
TABLE_DIR  = BASE_DIR / 'Tables'
MODEL_DIR  = BASE_DIR / 'Models'
OUTPUT_DIR = BASE_DIR / 'Outputs'
LOG_DIR    = BASE_DIR / 'logs'
PYTHON_DIR = BASE_DIR / 'Notebook'

for directory in [
    FIG_DIR,
    TABLE_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
    LOG_DIR,
    PYTHON_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

PRIMARY_DATA_PATH = Path('/content/drive/MyDrive/Datasets/Energy/Renewable.csv')
UNISOLAR_DIR = Path('/content/drive/MyDrive/Datasets/Energy/UniSolar')

print(f'Framework: {FRAMEWORK_NAME} — {FRAMEWORK_FULL_NAME}')
print(f'Base directory: {BASE_DIR}')
print(f'Primary dataset path: {PRIMARY_DATA_PATH}')
print(f'UniSolar directory: {UNISOLAR_DIR}')
print('Project directories created successfully.')


## 2. Imports, Logging, and Utility Functions

In [ ]:
# ============================================================
# 2. Imports, Logging, and Utility Functions
# ============================================================

import os
import sys
import json
import math
import shutil
import subprocess
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    explained_variance_score,
    max_error,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

LOG_PATH = LOG_DIR / 'outputs_summary.txt'

def log_message(message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{timestamp}] {message}'
    print(line)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def save_table(df, filename):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    log_message(f'Saved table: {path}')
    return path

def save_figure(filename, dpi=300):
    path = FIG_DIR / filename
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    log_message(f'Saved figure: {path}')
    return path

def normalize_columns(df):
    out = df.copy()
    out.columns = (
        out.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )
    return out

def detect_datetime_column(df):
    candidates = [
        c for c in df.columns
        if any(k in c for k in ['time', 'date', 'timestamp', 'datetime'])
    ]
    for c in candidates:
        parsed = pd.to_datetime(df[c], errors='coerce')
        if parsed.notna().mean() > 0.60:
            return c
    return None

def detect_target_column(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    priority_terms = [
        'energy_deltawh',
        'energy_delta_wh',
        'energy_delta',
        'solar_generation',
        'solargeneration',
        'generation',
        'energy',
        'power',
        'output',
        'yield'
    ]

    for term in priority_terms:
        for c in numeric_cols:
            if term in c:
                return c

    return numeric_cols[-1] if numeric_cols else None

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    naive = np.mean(np.abs(np.diff(y_true))) if len(y_true) > 1 else np.nan
    mase = mae / naive if naive and naive > 0 else np.nan

    smape = np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred) + 1e-9)
    ) * 100

    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': rmse,
        'MAE': mae,
        'MedAE': median_absolute_error(y_true, y_pred),
        'MaxError': max_error(y_true, y_pred),
        'ExplainedVariance': explained_variance_score(y_true, y_pred),
        'sMAPE_percent': smape,
        'NRMSE_range': rmse / (np.max(y_true) - np.min(y_true) + 1e-9),
        'NRMSE_std': rmse / (np.std(y_true) + 1e-9),
        'MASE': mase
    }

def classification_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'BalancedAccuracy': balanced_accuracy_score(y_true, y_pred),
        'F1_macro': f1_score(y_true, y_pred, average='macro'),
        'F1_weighted': f1_score(y_true, y_pred, average='weighted')
    }

log_message(f'Notebook initialized for {FRAMEWORK_NAME}.')


## 3. Load and Prepare Primary Dataset

In [ ]:
# ============================================================
# 3. Load and Prepare Primary Dataset
# ============================================================

if not PRIMARY_DATA_PATH.exists():
    raise FileNotFoundError(
        f'Primary dataset not found at {PRIMARY_DATA_PATH}. '
        'Please verify that Google Drive is mounted and the file exists.'
    )

primary_raw = pd.read_csv(PRIMARY_DATA_PATH)
primary_raw = normalize_columns(primary_raw)

log_message(f'Loaded primary raw dataset with shape: {primary_raw.shape}')

primary_target = detect_target_column(primary_raw)
primary_datetime = detect_datetime_column(primary_raw)

if primary_target is None:
    raise ValueError('Could not detect a numerical target column in the primary dataset.')

log_message(f'Primary target column detected: {primary_target}')

primary_df = primary_raw.copy()

if primary_datetime is not None:
    log_message(f'Primary datetime column detected: {primary_datetime}')
    primary_df[primary_datetime] = pd.to_datetime(primary_df[primary_datetime], errors='coerce')
    primary_df = primary_df.sort_values(primary_datetime)
    primary_df['hour'] = primary_df[primary_datetime].dt.hour
    primary_df['month'] = primary_df[primary_datetime].dt.month
    primary_df['dayofyear'] = primary_df[primary_datetime].dt.dayofyear
else:
    log_message('No datetime column detected in primary dataset. Row order is used chronologically.')

primary_df = primary_df.drop_duplicates()
primary_df[primary_target] = pd.to_numeric(primary_df[primary_target], errors='coerce')
primary_df = primary_df.dropna(subset=[primary_target])
primary_df = primary_df[primary_df[primary_target] >= 0].copy()

drop_cols = [primary_target]
if primary_datetime is not None:
    drop_cols.append(primary_datetime)

X_all = primary_df.drop(columns=drop_cols, errors='ignore')
y_all = primary_df[primary_target].copy()

for col in X_all.columns:
    if X_all[col].dtype == 'object':
        converted = pd.to_numeric(X_all[col], errors='coerce')
        if converted.notna().mean() > 0.80:
            X_all[col] = converted

missing_ratio = X_all.isna().mean()
X_all = X_all.loc[:, missing_ratio < 0.50]

for col in X_all.columns:
    if pd.api.types.is_numeric_dtype(X_all[col]):
        X_all[col] = X_all[col].fillna(X_all[col].median())
    else:
        X_all[col] = X_all[col].astype(str).fillna('missing')

log_message(f'Cleaned primary dataset shape: {primary_df.shape}')
log_message(f'Usable primary features: {X_all.shape[1]}')

display(primary_df.head())


## 4. Leakage-Safe Chronological Split

In [ ]:
# ============================================================
# 4. Leakage-Safe Chronological Split
# ============================================================

n = len(X_all)

train_end = int(0.60 * n)
val_end = int(0.80 * n)

X_train = X_all.iloc[:train_end].copy()
y_train = y_all.iloc[:train_end].copy()

X_val = X_all.iloc[train_end:val_end].copy()
y_val = y_all.iloc[train_end:val_end].copy()

X_test = X_all.iloc[val_end:].copy()
y_test = y_all.iloc[val_end:].copy()

X_trainval = X_all.iloc[:val_end].copy()
y_trainval = y_all.iloc[:val_end].copy()

numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numerical_features]

try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', encoder, categorical_features)
    ],
    remainder='drop'
)

log_message('Chronological split completed. Test set remains untouched.')
log_message(f'Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}')
log_message(f'Numerical features: {len(numerical_features)} | Categorical features: {len(categorical_features)}')


## 5. Primary Forecasting Model Selection

In [ ]:
# ============================================================
# 5. Primary Forecasting Model Selection
# ============================================================

RANDOM_STATE = 42

models = {
    'Dummy_Median': DummyRegressor(strategy='median'),
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'DecisionTree': DecisionTreeRegressor(random_state=RANDOM_STATE),
    'RandomForest': RandomForestRegressor(n_estimators=120, random_state=RANDOM_STATE, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=160, random_state=RANDOM_STATE, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingRegressor(random_state=RANDOM_STATE)
}

if HAS_XGBOOST:
    models['XGBoost'] = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='reg:squarederror',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

if HAS_LIGHTGBM:
    models['LightGBM'] = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    )

validation_rows = []

for name, model in models.items():
    log_message(f'Fitting validation model: {name}')

    pipe = Pipeline([
        ('preprocess', preprocess),
        ('model', clone(model))
    ])

    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)

    metrics = regression_metrics(y_val, val_pred)
    metrics['Model'] = name
    validation_rows.append(metrics)

validation_df = pd.DataFrame(validation_rows).sort_values('RMSE')
save_table(validation_df, 'validation_model_comparison.csv')

best_model_name = validation_df.iloc[0]['Model']
best_model = models[best_model_name]

log_message(f'Selected model based only on validation RMSE: {best_model_name}')

display(validation_df)


## 6. Final Primary Forecasting Evaluation

In [ ]:
# ============================================================
# 6. Final Primary Forecasting Evaluation
# ============================================================

final_model = Pipeline([
    ('preprocess', preprocess),
    ('model', clone(best_model))
])

final_model.fit(X_trainval, y_trainval)

test_pred = final_model.predict(X_test)
test_metrics = regression_metrics(y_test, test_pred)

internal_metrics_df = pd.DataFrame([test_metrics])
save_table(internal_metrics_df, 'internal_test_metrics.csv')

predictions_df = pd.DataFrame({
    'Actual_Wh': y_test.values,
    'Predicted_Wh': test_pred,
    'Residual_Wh': y_test.values - test_pred,
    'Absolute_Error_Wh': np.abs(y_test.values - test_pred)
})

save_table(predictions_df, 'internal_test_predictions.csv')

import joblib
model_path = MODEL_DIR / 'final_virem_primary_model.pkl'
joblib.dump(final_model, model_path)

log_message(f'Final model saved: {model_path}')
log_message(f'Final internal test metrics: {json.dumps(test_metrics, indent=2)}')

display(internal_metrics_df)


## 7. UniSolar Discovery and External Harmonization

In [ ]:
# ============================================================
# 7. UniSolar Discovery and External Harmonization
# ============================================================

if not UNISOLAR_DIR.exists():
    raise FileNotFoundError(
        f'UniSolar directory not found at {UNISOLAR_DIR}. '
        'Please verify that the folder exists.'
    )

unisolar_files = sorted(list(UNISOLAR_DIR.glob('*.csv')))
log_message(f'Found {len(unisolar_files)} UniSolar CSV files.')

for f in unisolar_files:
    log_message(f'UniSolar file: {f.name}')

loaded_unisolar = {}

for f in unisolar_files:
    df = pd.read_csv(f)
    df = normalize_columns(df)
    loaded_unisolar[f.name] = df
    log_message(f'Loaded {f.name} with shape {df.shape}')

generation_name = None
weather_name = None
site_name = None
summary_name = None

for name in loaded_unisolar:
    low = name.lower()
    if 'generation' in low:
        generation_name = name
    elif 'weather' in low:
        weather_name = name
    elif 'site' in low:
        site_name = name
    elif 'summary' in low or 'monthly' in low:
        summary_name = name

if generation_name is None:
    raise ValueError('Could not infer UniSolar generation file.')

if weather_name is None:
    raise ValueError('Could not infer UniSolar weather file.')

gen_df = loaded_unisolar[generation_name].copy()
weather_df = loaded_unisolar[weather_name].copy()

log_message(f'Inferred generation file: {generation_name}')
log_message(f'Inferred weather file: {weather_name}')
log_message(f'Inferred site file: {site_name}')
log_message(f'Inferred summary file: {summary_name}')

display(gen_df.head())
display(weather_df.head())


## 8. UniSolar External Raw Prediction Diagnostic

In [ ]:
# ============================================================
# 8. UniSolar External Raw Prediction Diagnostic
# ============================================================

gen_time = detect_datetime_column(gen_df)
weather_time = detect_datetime_column(weather_df)

if gen_time is None:
    raise ValueError('Could not detect datetime column in UniSolar generation file.')

if weather_time is None:
    raise ValueError('Could not detect datetime column in UniSolar weather file.')

gen_target = detect_target_column(gen_df)

if gen_target is None:
    raise ValueError('Could not detect target column in UniSolar generation file.')

log_message(f'UniSolar datetime columns: generation={gen_time}, weather={weather_time}')
log_message(f'UniSolar target column detected: {gen_target}')

gen_df[gen_time] = pd.to_datetime(gen_df[gen_time], errors='coerce')
weather_df[weather_time] = pd.to_datetime(weather_df[weather_time], errors='coerce')

gen_df = gen_df.dropna(subset=[gen_time, gen_target]).copy()
weather_df = weather_df.dropna(subset=[weather_time]).copy()

gen_site_candidates = [c for c in gen_df.columns if any(k in c for k in ['site', 'sitekey', 'plant', 'location'])]
weather_site_candidates = [c for c in weather_df.columns if any(k in c for k in ['site', 'sitekey', 'plant', 'location'])]

gen_site = gen_site_candidates[0] if gen_site_candidates else None
weather_site = weather_site_candidates[0] if weather_site_candidates else None

log_message(f'UniSolar site columns: generation={gen_site}, weather={weather_site}')

if gen_site and weather_site:
    gen_agg = gen_df.groupby([gen_time, gen_site], as_index=False)[gen_target].mean()
    weather_num_cols = weather_df.select_dtypes(include=[np.number]).columns.tolist()
    weather_agg = weather_df.groupby([weather_time, weather_site], as_index=False)[weather_num_cols].mean()

    uni = pd.merge(
        gen_agg,
        weather_agg,
        left_on=[gen_time, gen_site],
        right_on=[weather_time, weather_site],
        how='inner'
    )
else:
    gen_agg = gen_df.groupby(gen_time, as_index=False)[gen_target].mean()
    weather_num_cols = weather_df.select_dtypes(include=[np.number]).columns.tolist()
    weather_agg = weather_df.groupby(weather_time, as_index=False)[weather_num_cols].mean()

    uni = pd.merge(
        gen_agg,
        weather_agg,
        left_on=gen_time,
        right_on=weather_time,
        how='inner'
    )

log_message(f'UniSolar merged dataset shape: {uni.shape}')

uni[gen_target] = pd.to_numeric(uni[gen_target], errors='coerce')
uni = uni.dropna(subset=[gen_target])
uni = uni[uni[gen_target] >= 0].copy()

uni['hour'] = pd.to_datetime(uni[gen_time], errors='coerce').dt.hour
uni['month'] = pd.to_datetime(uni[gen_time], errors='coerce').dt.month
uni['dayofyear'] = pd.to_datetime(uni[gen_time], errors='coerce').dt.dayofyear

X_external = pd.DataFrame(index=uni.index)

for col in X_trainval.columns:
    if col in uni.columns:
        X_external[col] = uni[col]
    else:
        X_external[col] = 0

for col in X_external.columns:
    if col in X_trainval.columns and pd.api.types.is_numeric_dtype(X_trainval[col]):
        X_external[col] = pd.to_numeric(X_external[col], errors='coerce')
        X_external[col] = X_external[col].fillna(X_trainval[col].median())
    else:
        X_external[col] = X_external[col].astype(str).fillna('missing')

y_external = uni[gen_target].copy()

MAX_EXTERNAL_ROWS = 200000

if len(X_external) > MAX_EXTERNAL_ROWS:
    sample_idx = np.linspace(0, len(X_external) - 1, MAX_EXTERNAL_ROWS).astype(int)
    X_external_eval = X_external.iloc[sample_idx].copy()
    y_external_eval = y_external.iloc[sample_idx].copy()
    uni_eval = uni.iloc[sample_idx].copy()
    log_message(f'UniSolar external evaluation sampled to {MAX_EXTERNAL_ROWS} rows.')
else:
    X_external_eval = X_external
    y_external_eval = y_external
    uni_eval = uni.copy()

external_pred = final_model.predict(X_external_eval)
external_metrics = regression_metrics(y_external_eval, external_pred)

external_metrics_df = pd.DataFrame([external_metrics])
save_table(external_metrics_df, 'external_unisolar_raw_prediction_metrics.csv')

comparison_df = pd.concat([
    internal_metrics_df.assign(Dataset='Primary_Internal_Test'),
    external_metrics_df.assign(Dataset='UniSolar_External_Raw_Target')
], ignore_index=True)

save_table(comparison_df, 'internal_external_raw_prediction_comparison.csv')

log_message(f'External UniSolar raw prediction metrics: {json.dumps(external_metrics, indent=2)}')

display(comparison_df)


## 9. Distribution-Shift Diagnostics

In [ ]:
# ============================================================
# 9. Distribution-Shift Diagnostics
# ============================================================

shift_rows = []

for col in X_trainval.columns:
    if col in X_external_eval.columns and pd.api.types.is_numeric_dtype(X_trainval[col]):
        a = pd.to_numeric(X_trainval[col], errors='coerce').dropna()
        b = pd.to_numeric(X_external_eval[col], errors='coerce').dropna()

        if len(a) > 10 and len(b) > 10:
            shift_rows.append({
                'Feature': col,
                'Primary_Mean': a.mean(),
                'External_Mean': b.mean(),
                'Primary_Std': a.std(),
                'External_Std': b.std(),
                'Mean_Shift_StdUnits': abs(a.mean() - b.mean()) / (a.std() + 1e-9),
                'Primary_Q05': a.quantile(0.05),
                'External_Q05': b.quantile(0.05),
                'Primary_Q95': a.quantile(0.95),
                'External_Q95': b.quantile(0.95)
            })

shift_df = pd.DataFrame(shift_rows).sort_values('Mean_Shift_StdUnits', ascending=False)
save_table(shift_df, 'feature_distribution_shift_diagnostics.csv')

global_shift_score = float(
    shift_df['Mean_Shift_StdUnits']
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .mean()
)

log_message(f'Global distribution-shift score: {global_shift_score:.4f}')

plt.figure(figsize=(10, 5))
top_shift = shift_df.head(15)
plt.bar(top_shift['Feature'], top_shift['Mean_Shift_StdUnits'])
plt.xticks(rotation=75, ha='right')
plt.ylabel('Mean shift in primary standard-deviation units')
plt.title('Top Feature Distribution Shifts: Primary vs UniSolar')
save_figure('top_feature_distribution_shifts.png')
plt.show()

display(shift_df.head(20))


## 10. Latent Operational-Regime Construction

In [ ]:
# ============================================================
# 10. Latent Operational-Regime Construction
# ============================================================

REGIME_ORDER = [
    'Critical_Low',
    'Low',
    'Normal',
    'High',
    'Surplus'
]

def assign_operational_regime(values, reference_values=None):
    values = pd.Series(values).astype(float)

    if reference_values is None:
        reference_values = values
    else:
        reference_values = pd.Series(reference_values).astype(float)

    q10 = reference_values.quantile(0.10)
    q30 = reference_values.quantile(0.30)
    q70 = reference_values.quantile(0.70)
    q90 = reference_values.quantile(0.90)

    def label(v):
        if v <= q10:
            return 'Critical_Low'
        elif v <= q30:
            return 'Low'
        elif v <= q70:
            return 'Normal'
        elif v <= q90:
            return 'High'
        else:
            return 'Surplus'

    return values.apply(label), {
        'q10': q10,
        'q30': q30,
        'q70': q70,
        'q90': q90
    }

# Internal actual and predicted regimes use primary target reference.
internal_actual_regime, primary_regime_thresholds = assign_operational_regime(
    y_test.values,
    reference_values=y_trainval.values
)

internal_pred_regime, _ = assign_operational_regime(
    test_pred,
    reference_values=y_trainval.values
)

# External actual and predicted regimes use UniSolar's own target reference.
# This is the key latent alignment correction.
external_actual_regime, external_regime_thresholds = assign_operational_regime(
    y_external_eval.values,
    reference_values=y_external_eval.values
)

external_pred_regime, _ = assign_operational_regime(
    external_pred,
    reference_values=external_pred
)

regime_thresholds_df = pd.DataFrame([
    {'Dataset': 'Primary', **primary_regime_thresholds},
    {'Dataset': 'UniSolar', **external_regime_thresholds}
])

save_table(regime_thresholds_df, 'latent_regime_thresholds.csv')

internal_regime_df = pd.DataFrame({
    'Dataset': 'Primary_Internal_Test',
    'ActualValue': y_test.values,
    'PredictedValue': test_pred,
    'ActualRegime': internal_actual_regime.values,
    'PredictedRegime': internal_pred_regime.values
})

external_regime_df = pd.DataFrame({
    'Dataset': 'UniSolar_External_Test',
    'ActualValue': y_external_eval.values,
    'PredictedValue': external_pred,
    'ActualRegime': external_actual_regime.values,
    'PredictedRegime': external_pred_regime.values
})

save_table(internal_regime_df, 'internal_latent_regimes.csv')
save_table(external_regime_df, 'external_latent_regimes.csv')

internal_regime_metrics = classification_metrics(
    internal_regime_df['ActualRegime'],
    internal_regime_df['PredictedRegime']
)

external_regime_metrics = classification_metrics(
    external_regime_df['ActualRegime'],
    external_regime_df['PredictedRegime']
)

regime_metrics_df = pd.DataFrame([
    {'Dataset': 'Primary_Internal_Test', **internal_regime_metrics},
    {'Dataset': 'UniSolar_External_Test', **external_regime_metrics}
])

save_table(regime_metrics_df, 'latent_regime_alignment_metrics.csv')

log_message(f'Internal latent-regime metrics: {json.dumps(internal_regime_metrics, indent=2)}')
log_message(f'External latent-regime metrics: {json.dumps(external_regime_metrics, indent=2)}')

display(regime_metrics_df)


## 11. Regime Distribution and Confusion Matrices

In [ ]:
# ============================================================
# 11. Regime Distribution and Confusion Matrices
# ============================================================

combined_regime_df = pd.concat([
    internal_regime_df,
    external_regime_df
], ignore_index=True)

regime_distribution = (
    combined_regime_df
    .groupby(['Dataset', 'ActualRegime'])
    .size()
    .reset_index(name='Count')
)

regime_distribution['Percent'] = regime_distribution.groupby('Dataset')['Count'].transform(
    lambda s: 100 * s / s.sum()
)

save_table(regime_distribution, 'latent_regime_distribution.csv')

for dataset_name, df_part in combined_regime_df.groupby('Dataset'):
    cm = confusion_matrix(
        df_part['ActualRegime'],
        df_part['PredictedRegime'],
        labels=REGIME_ORDER
    )

    cm_df = pd.DataFrame(cm, index=REGIME_ORDER, columns=REGIME_ORDER)
    save_table(cm_df.reset_index().rename(columns={'index': 'ActualRegime'}),
               f'confusion_matrix_{dataset_name}.csv')

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, aspect='auto')
    plt.xticks(range(len(REGIME_ORDER)), REGIME_ORDER, rotation=45, ha='right')
    plt.yticks(range(len(REGIME_ORDER)), REGIME_ORDER)
    plt.title(f'Latent Regime Confusion Matrix — {dataset_name}')
    plt.xlabel('Predicted regime')
    plt.ylabel('Actual regime')
    plt.colorbar(label='Count')
    save_figure(f'latent_regime_confusion_{dataset_name}.png')
    plt.show()

plt.figure(figsize=(10, 5))
for dataset_name in regime_distribution['Dataset'].unique():
    subset = regime_distribution[regime_distribution['Dataset'] == dataset_name]
    subset = subset.set_index('ActualRegime').reindex(REGIME_ORDER).fillna(0)
    plt.plot(REGIME_ORDER, subset['Percent'], marker='o', label=dataset_name)

plt.ylabel('Actual regime distribution (%)')
plt.title('Latent Operational-Regime Distribution Across Datasets')
plt.xticks(rotation=30, ha='right')
plt.legend()
save_figure('latent_regime_distribution_comparison.png')
plt.show()

display(regime_distribution)


## 12. Regime Transition Analysis

In [ ]:
# ============================================================
# 12. Regime Transition Analysis
# ============================================================

def compute_regime_transitions(regime_series, dataset_name):
    s = pd.Series(regime_series).reset_index(drop=True)
    transitions = pd.DataFrame({
        'FromRegime': s.iloc[:-1].values,
        'ToRegime': s.iloc[1:].values
    })

    transitions['Dataset'] = dataset_name
    transitions['Changed'] = transitions['FromRegime'] != transitions['ToRegime']

    return transitions

internal_transitions = compute_regime_transitions(
    internal_regime_df['ActualRegime'],
    'Primary_Internal_Test'
)

external_transitions = compute_regime_transitions(
    external_regime_df['ActualRegime'],
    'UniSolar_External_Test'
)

all_transitions = pd.concat([
    internal_transitions,
    external_transitions
], ignore_index=True)

transition_summary = (
    all_transitions
    .groupby('Dataset')
    .agg(
        TransitionCount=('Changed', 'count'),
        RegimeChangePercent=('Changed', lambda x: 100 * x.mean())
    )
    .reset_index()
)

transition_matrix = (
    all_transitions
    .groupby(['Dataset', 'FromRegime', 'ToRegime'])
    .size()
    .reset_index(name='Count')
)

save_table(all_transitions, 'latent_regime_transitions_samplewise.csv')
save_table(transition_summary, 'latent_regime_transition_summary.csv')
save_table(transition_matrix, 'latent_regime_transition_matrix_long.csv')

plt.figure(figsize=(8, 4))
plt.bar(transition_summary['Dataset'], transition_summary['RegimeChangePercent'])
plt.ylabel('Regime-change frequency (%)')
plt.title('Operational Regime Volatility Across Datasets')
plt.xticks(rotation=20, ha='right')
save_figure('latent_regime_transition_frequency.png')
plt.show()

display(transition_summary)


## 13. Regime-Aware VIREM-UVIF Decision Layer

In [ ]:
# ============================================================
# 13. Regime-Aware VIREM-UVIF Decision Layer
# ============================================================

regime_stress_map = {
    'Critical_Low': 1.00,
    'Low': 0.70,
    'Normal': 0.25,
    'High': 0.35,
    'Surplus': 0.55
}

regime_information_need_map = {
    'Critical_Low': 0.70,
    'Low': 0.45,
    'Normal': 0.15,
    'High': 0.30,
    'Surplus': 0.40
}

ACTION_LIBRARY = {
    'Normal_Operation':       {'stress_reduction': 0.05, 'cost': 0.05, 'information_gain': 0.10},
    'Load_Reduction':         {'stress_reduction': 0.45, 'cost': 0.35, 'information_gain': 0.25},
    'Battery_Activation':     {'stress_reduction': 0.65, 'cost': 0.45, 'information_gain': 0.30},
    'Energy_Storage':         {'stress_reduction': 0.30, 'cost': 0.25, 'information_gain': 0.20},
    'Evidence_Acquisition':   {'stress_reduction': 0.15, 'cost': 0.20, 'information_gain': 0.80},
    'Emergency_Protection':   {'stress_reduction': 0.90, 'cost': 0.75, 'information_gain': 0.45}
}

def regime_baseline_decision(regime):
    if regime == 'Critical_Low':
        return 'Battery_Activation'
    if regime == 'Low':
        return 'Load_Reduction'
    if regime == 'Normal':
        return 'Normal_Operation'
    if regime == 'High':
        return 'Normal_Operation'
    if regime == 'Surplus':
        return 'Energy_Storage'
    return 'Evidence_Acquisition'

def adaptive_regime_uvif_weights(regime, uncertainty, shift_score):
    stress = regime_stress_map.get(regime, 0.5)
    information_need = regime_information_need_map.get(regime, 0.5)

    return {
        'lambda_stress': 0.75 + stress,
        'lambda_uncertainty': 0.50 + uncertainty,
        'lambda_shift': 0.50 + shift_score,
        'lambda_cost': 0.25 + stress,
        'lambda_information': 1.00 + information_need + shift_score
    }

def regime_uvif_score(regime, uncertainty, shift_score, action_params):
    weights = adaptive_regime_uvif_weights(regime, uncertainty, shift_score)
    stress = regime_stress_map.get(regime, 0.5)
    information_need = regime_information_need_map.get(regime, 0.5)

    score = (
        weights['lambda_stress'] * stress
        + weights['lambda_uncertainty'] * uncertainty
        + weights['lambda_shift'] * shift_score
        + weights['lambda_cost'] * action_params['cost']
        - weights['lambda_information'] * action_params['information_gain']
        - action_params['stress_reduction']
        - 0.25 * information_need
    )

    return score

def select_regime_uvif_actions(regimes, uncertainty, shift_score):
    selected = []
    score_rows = []

    for i, regime in enumerate(regimes):
        u = float(uncertainty[i])
        scores = {}

        for action, params in ACTION_LIBRARY.items():
            scores[action] = regime_uvif_score(
                regime=regime,
                uncertainty=u,
                shift_score=shift_score,
                action_params=params
            )

        best_action = min(scores, key=scores.get)
        selected.append(best_action)

        row = {
            'Sample': i,
            'Regime': regime,
            'Uncertainty': u,
            'ShiftScore': shift_score,
            'SelectedAction': best_action
        }

        for action, score in scores.items():
            row[f'Score_{action}'] = score

        score_rows.append(row)

    return np.array(selected), pd.DataFrame(score_rows)

# Uncertainty proxy: absolute normalized residual for internal, model spread proxy fallback for external.
internal_regime_uncertainty = np.clip(
    np.abs(y_test.values - test_pred) / (np.percentile(y_trainval, 95) + 1e-9),
    0,
    2
)

external_regime_uncertainty = np.clip(
    np.abs(y_external_eval.values - external_pred) / (np.percentile(y_external_eval, 95) + 1e-9),
    0,
    2
)

internal_regime_actions, internal_regime_scores = select_regime_uvif_actions(
    internal_regime_df['PredictedRegime'].values,
    internal_regime_uncertainty,
    shift_score=0.0
)

external_regime_actions, external_regime_scores = select_regime_uvif_actions(
    external_regime_df['PredictedRegime'].values,
    external_regime_uncertainty,
    shift_score=global_shift_score
)

internal_regime_decisions = internal_regime_df.copy()
internal_regime_decisions['Uncertainty'] = internal_regime_uncertainty
internal_regime_decisions['BaselineDecision'] = internal_regime_decisions['PredictedRegime'].apply(regime_baseline_decision)
internal_regime_decisions['VIREM_UVIF_RegimeDecision'] = internal_regime_actions

external_regime_decisions = external_regime_df.copy()
external_regime_decisions['Uncertainty'] = external_regime_uncertainty
external_regime_decisions['BaselineDecision'] = external_regime_decisions['PredictedRegime'].apply(regime_baseline_decision)
external_regime_decisions['VIREM_UVIF_RegimeDecision'] = external_regime_actions

save_table(internal_regime_decisions, 'internal_regime_aware_virem_decisions.csv')
save_table(external_regime_decisions, 'external_regime_aware_virem_decisions.csv')
save_table(internal_regime_scores, 'internal_regime_aware_uvif_scores.csv')
save_table(external_regime_scores, 'external_regime_aware_uvif_scores.csv')

regime_decision_distribution = pd.concat([
    internal_regime_decisions.groupby(['Dataset', 'VIREM_UVIF_RegimeDecision']).size().reset_index(name='Count'),
    external_regime_decisions.groupby(['Dataset', 'VIREM_UVIF_RegimeDecision']).size().reset_index(name='Count')
], ignore_index=True)

regime_decision_distribution['Percent'] = regime_decision_distribution.groupby('Dataset')['Count'].transform(
    lambda s: 100 * s / s.sum()
)

save_table(regime_decision_distribution, 'regime_aware_virem_decision_distribution.csv')

display(regime_decision_distribution)


## 14. Decision Robustness Under Latent Regime Alignment

In [ ]:
# ============================================================
# 14. Decision Robustness Under Latent Regime Alignment
# ============================================================

def decision_robustness_summary(df):
    return {
        'Dataset': df['Dataset'].iloc[0],
        'Samples': len(df),
        'MeanUncertainty': df['Uncertainty'].mean(),
        'EmergencyProtectionPercent': (df['VIREM_UVIF_RegimeDecision'] == 'Emergency_Protection').mean() * 100,
        'EvidenceAcquisitionPercent': (df['VIREM_UVIF_RegimeDecision'] == 'Evidence_Acquisition').mean() * 100,
        'NormalOperationPercent': (df['VIREM_UVIF_RegimeDecision'] == 'Normal_Operation').mean() * 100,
        'BaselineUVIFDisagreementPercent': (
            df['BaselineDecision'] != df['VIREM_UVIF_RegimeDecision']
        ).mean() * 100,
        'CriticalLowPercent': (df['PredictedRegime'] == 'Critical_Low').mean() * 100,
        'SurplusPercent': (df['PredictedRegime'] == 'Surplus').mean() * 100
    }

regime_robustness_df = pd.DataFrame([
    decision_robustness_summary(internal_regime_decisions),
    decision_robustness_summary(external_regime_decisions)
])

save_table(regime_robustness_df, 'regime_aware_decision_robustness_summary.csv')

plt.figure(figsize=(8, 4))
plt.bar(regime_robustness_df['Dataset'], regime_robustness_df['BaselineUVIFDisagreementPercent'])
plt.ylabel('Baseline–UVIF disagreement (%)')
plt.title('Regime-Aware Decision Reconfiguration Under Shift')
plt.xticks(rotation=20, ha='right')
save_figure('regime_aware_decision_reconfiguration.png')
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(regime_robustness_df['Dataset'], regime_robustness_df['EvidenceAcquisitionPercent'])
plt.ylabel('Evidence acquisition (%)')
plt.title('Evidence-Seeking Behavior After Latent Regime Alignment')
plt.xticks(rotation=20, ha='right')
save_figure('regime_aware_evidence_acquisition.png')
plt.show()

display(regime_robustness_df)


## 15. Comparative Scientific Summary

In [ ]:
# ============================================================
# 15. Comparative Scientific Summary
# ============================================================

scientific_summary = pd.DataFrame([
    {
        'Layer': 'Raw regression transfer',
        'InternalFinding': f"R2 = {test_metrics['R2']:.4f}",
        'ExternalFinding': f"R2 = {external_metrics['R2']:.4f}",
        'Interpretation': 'Raw targets are not physically equivalent across datasets.'
    },
    {
        'Layer': 'Distribution-shift diagnosis',
        'InternalFinding': 'Primary training environment',
        'ExternalFinding': f"Global shift score = {global_shift_score:.4f}",
        'Interpretation': 'External dataset exhibits severe feature and operational mismatch.'
    },
    {
        'Layer': 'Latent regime alignment',
        'InternalFinding': f"Balanced accuracy = {internal_regime_metrics['BalancedAccuracy']:.4f}",
        'ExternalFinding': f"Balanced accuracy = {external_regime_metrics['BalancedAccuracy']:.4f}",
        'Interpretation': 'Operational regimes provide a more meaningful cross-dataset comparison than raw target transfer.'
    },
    {
        'Layer': 'Regime-aware VIREM-UVIF',
        'InternalFinding': f"Evidence acquisition = {regime_robustness_df.iloc[0]['EvidenceAcquisitionPercent']:.2f}%",
        'ExternalFinding': f"Evidence acquisition = {regime_robustness_df.iloc[1]['EvidenceAcquisitionPercent']:.2f}%",
        'Interpretation': 'The variational decision layer reconfigures actions in response to uncertainty and shift.'
    }
])

save_table(scientific_summary, 'scientific_summary_for_manuscript.csv')

display(scientific_summary)


## 16. Figures for Manuscript-Level Reporting

In [ ]:
# ============================================================
# 16. Figures for Manuscript-Level Reporting
# ============================================================

plt.figure(figsize=(8, 4))
plt.bar(
    regime_metrics_df['Dataset'],
    regime_metrics_df['BalancedAccuracy']
)
plt.ylabel('Balanced accuracy')
plt.title('Latent Operational-Regime Alignment Performance')
plt.xticks(rotation=20, ha='right')
save_figure('latent_regime_alignment_balanced_accuracy.png')
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(
    regime_robustness_df['Dataset'],
    regime_robustness_df['BaselineUVIFDisagreementPercent']
)
plt.ylabel('Decision reconfiguration (%)')
plt.title('VIREM-UVIF Decision Reconfiguration Across Domains')
plt.xticks(rotation=20, ha='right')
save_figure('virem_uvif_decision_reconfiguration_across_domains.png')
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(
    transition_summary['Dataset'],
    transition_summary['RegimeChangePercent']
)
plt.ylabel('Regime-transition frequency (%)')
plt.title('Operational Volatility Across Renewable Datasets')
plt.xticks(rotation=20, ha='right')
save_figure('operational_volatility_across_datasets.png')
plt.show()


## 17. Notebook Destination and Reproducibility Manifest

In [ ]:
# ============================================================
# 17. Notebook Destination and Reproducibility Manifest
# ============================================================

manifest = {
    'framework_abbreviation': FRAMEWORK_NAME,
    'framework_full_name': FRAMEWORK_FULL_NAME,
    'base_directory': str(BASE_DIR),
    'primary_dataset': str(PRIMARY_DATA_PATH),
    'unisolar_directory': str(UNISOLAR_DIR),
    'selected_model': str(best_model_name),
    'internal_raw_regression_metrics': test_metrics,
    'external_raw_regression_metrics': external_metrics,
    'global_distribution_shift_score': global_shift_score,
    'internal_regime_metrics': internal_regime_metrics,
    'external_regime_metrics': external_regime_metrics,
    'created_at': datetime.now().isoformat(),
    'notebook_destination': str(PYTHON_DIR / 'VIREM_UVIF_Latent_Regime_Alignment_Q1.ipynb'),
    'scientific_note': (
        'This notebook reframes cross-dataset renewable-energy analysis from raw target transfer '
        'to latent operational-regime alignment, allowing VIREM-UVIF to evaluate decision robustness '
        'under severe distribution shift.'
    )
}

manifest_path = OUTPUT_DIR / 'reproducibility_manifest.json'

with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

log_message(f'Reproducibility manifest saved: {manifest_path}')

NOTEBOOK_DESTINATION = PYTHON_DIR / 'VIREM_UVIF_Latent_Regime_Alignment_Q1.ipynb'
log_message(f'Notebook destination required: {NOTEBOOK_DESTINATION}')

print('Notebook completed successfully.')
print(f'Please ensure the notebook itself is saved to: {NOTEBOOK_DESTINATION}')
